# MLP and ResMLP

Three architectures per family, each trained with the same three learning rates, on every dataset size and data fraction.

In [1]:
import itertools
import os
import random

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

SEED = 42
NS = (20, 30, 40)
FRACS = (0.25, 0.5, 1.0)
LRS = (1e-3, 3e-3, 1e-2)  # the same three learning rates for every architecture
BATCH_SIZE = 128
MAX_EPOCHS = 300
PATIENCE = 40              # stop after this many epochs without a better val accuracy
WEIGHT_ROOT = "./weights"


def set_seed(seed=42):
    """Set a seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## Data

In [2]:
def load(name):
    """Load (X, y) arrays for a dataset."""
    return np.load(f"Datasets/{name}-X.npy"), np.load(f"Datasets/{name}-y.npy")


def split(X, y):
    """60% train, 20% validation, 20% test."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)
    return X_train, y_train, X_val, y_val, X_test, y_test


def binarise(X):
    """Map each noisy +-1 feature to its sign."""
    return np.where(X > 0, 1.0, -1.0)


def to_tensors(X, y):
    """Convert numpy arrays to float tensors."""
    return torch.from_numpy(X).float(), torch.from_numpy(y).float()

## Feature selection

In [3]:
def lookup_table_acc(S_train, y_train, S_val, y_val, cols):
    """Val accuracy of a majority-vote lookup table over the bits in `cols`."""
    w = 2 ** np.arange(len(cols))
    idx_tr = ((S_train[:, cols] > 0) * w).sum(1)
    idx_va = ((S_val[:, cols] > 0) * w).sum(1)
    count = np.bincount(idx_tr, minlength=2 ** len(cols))
    pos = np.bincount(idx_tr, weights=y_train, minlength=2 ** len(cols))
    table = (2 * pos > count).astype(int)  # unseen cells default to 0
    return (table[idx_va] == y_val).mean()


def select_features(S_train, y_train, S_val, y_val, min_gain=0.005):
    """Greedy forward selection that may add one bit or a pair of bits per step."""
    n = S_train.shape[1]
    score = lambda cols: lookup_table_acc(S_train, y_train, S_val, y_val, list(cols))
    selected, best = [], 0.5
    while True:
        rest = [j for j in range(n) if j not in selected]
        cands = [[j] for j in rest] + [list(p) for p in itertools.combinations(rest, 2)]
        acc, add = max((score(selected + c), c) for c in cands)
        if acc <= best + min_gain:
            break
        selected, best = selected + add, acc
        print(f"  + {add} -> {sorted(selected)}  val acc {acc:.4f}")
    if not selected:
        print("  no feature set beats chance; keeping all features")
        return list(range(n))
    return sorted(selected)

| Family | Size   | Hidden layers     |
|--------|--------|-------------------|
| MLP    | small  | (32,)             |
| MLP    | medium | (64, 64)          |
| MLP    | large  | (128, 128, 128)   |
| ResMLP | small  | (32,)             |
| ResMLP | medium | (64, 64)          |
| ResMLP | large  | (128, 128, 128)   |

In [4]:
class MLP(nn.Module):
    """Plain ReLU MLP that outputs one logit per row."""

    def __init__(self, d_in, hidden):
        super().__init__()
        layers, d = [], d_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU()]
            d = h
        self.net = nn.Sequential(*layers, nn.Linear(d, 1))

    def forward(self, x):
        return self.net(x).squeeze(1)


class ResBlock(nn.Module):
    """Pre-norm residual block: x + Linear(ReLU(Linear(LayerNorm(x))))."""

    def __init__(self, width):
        super().__init__()
        self.f = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))

    def forward(self, x):
        return x + self.f(x)


class ResMLP(nn.Module):
    """Linear stem, `blocks` residual blocks of size `width`, then a linear head."""

    def __init__(self, d_in, width, blocks):
        super().__init__()
        self.stem = nn.Linear(d_in, width)
        self.blocks = nn.Sequential(*[ResBlock(width) for _ in range(blocks)])
        self.head = nn.Sequential(nn.LayerNorm(width), nn.ReLU(), nn.Linear(width, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(1)


# family -> list of (architecture name, kwargs for the constructor)
FAMILIES = {
    "mlp": (MLP, [("small", dict(hidden=(32,))),
                  ("medium", dict(hidden=(64, 64))),
                  ("large", dict(hidden=(128, 128, 128)))]),
    "resmlp": (ResMLP, [("small", dict(width=32, blocks=1)),
                        ("medium", dict(width=64, blocks=2)),
                        ("large", dict(width=128, blocks=3))]),
}

for family, (cls, archs) in FAMILIES.items():
    for arch, kw in archs:
        n_params = sum(p.numel() for p in cls(10, **kw).parameters())
        print(f"{family:<7} {arch:<7} {kw}  {n_params:,} parameters for 10 inputs")


def accuracy(net, X, y):
    """Accuracy of `net` on tensors `X`, `y`."""
    net.eval()
    with torch.no_grad():
        return float(((net(X) > 0).float() == y).float().mean())


def train(net, X_tr, y_tr, X_va, y_va, lr):
    """Adam + BCE with early stopping on val accuracy; returns (best val acc, best epoch)."""
    set_seed(SEED)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    gen = torch.Generator().manual_seed(SEED)
    best_acc, best_ep, best_state = -1.0, 0, None
    for ep in range(1, MAX_EPOCHS + 1):
        net.train()
        for idx in torch.randperm(len(y_tr), generator=gen).split(BATCH_SIZE):
            loss = loss_fn(net(X_tr[idx]), y_tr[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
        acc = accuracy(net, X_va, y_va)
        if acc > best_acc:
            best_acc, best_ep = acc, ep
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        elif ep - best_ep >= PATIENCE:
            break
    net.load_state_dict(best_state)
    return best_acc, best_ep


def tune(family, train_data, val_data):
    """Train every architecture with every lr in LRS; keep the best on val (first wins ties)."""
    cls, archs = FAMILIES[family]
    d_in = train_data[0].shape[1]
    best = None
    for arch, kw in archs:
        for lr in LRS:
            set_seed(SEED)
            net = cls(d_in, **kw)
            val_acc, ep = train(net, *train_data, *val_data, lr)
            print(f"    {family:<7} {arch:<7} lr {lr:<6g} val {val_acc:.4f}  (best epoch {ep})")
            if best is None or val_acc > best[0]:
                best = (val_acc, dict(arch=arch, **kw, lr=lr, epoch=ep), net)
    return best

mlp     small   {'hidden': (32,)}  385 parameters for 10 inputs
mlp     medium  {'hidden': (64, 64)}  4,929 parameters for 10 inputs
mlp     large   {'hidden': (128, 128, 128)}  34,561 parameters for 10 inputs
resmlp  small   {'width': 32, 'blocks': 1}  2,625 parameters for 10 inputs
resmlp  medium  {'width': 64, 'blocks': 2}  17,793 parameters for 10 inputs
resmlp  large   {'width': 128, 'blocks': 3}  101,633 parameters for 10 inputs


## All datasets and scales

In [5]:
def run(n, frac):
    """Feature selection on the first `frac` of train/val, tune both families, save each best model."""
    X_train, y_train, X_val, y_val, X_test, y_test = split(*load(f"kryptonite-{n}"))
    m, mv = int(frac * len(y_train)), int(frac * len(y_val))
    S_tr, y_tr = binarise(X_train[:m]), y_train[:m]
    S_va, y_va = binarise(X_val[:mv]), y_val[:mv]
    S_te, y_te = binarise(X_test), y_test
    print(f"\n=== n={n}, {frac:.0%}: {m} train / {mv} val / {len(y_te)} test ===")

    cols = select_features(S_tr, y_tr, S_va, y_va)
    print(f"  selected {len(cols)} features: {cols}")
    tr = to_tensors(S_tr[:, cols], y_tr)
    va = to_tensors(S_va[:, cols], y_va)
    te = to_tensors(S_te[:, cols], y_te)

    rows = []
    for family in FAMILIES:
        val_acc, params, net = tune(family, tr, va)
        test_acc = accuracy(net, *te)
        out_dir = os.path.join(WEIGHT_ROOT, family, f"n{n}", f"{round(frac * 100)}pct")
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, "model.pt")
        torch.save({"state_dict": net.state_dict(), "features": cols, "params": params}, path)
        print(f"  best {family:<7} {params}  val {val_acc:.4f}  test {test_acc:.4f}  -> {path}")
        rows.append({"n": n, "frac": frac, "model": family, "train_size": m, "features": cols,
                     "params": params, "val": val_acc, "test": test_acc, "path": path})
    return rows


results = [row for n in NS for frac in FRACS for row in run(n, frac)]


=== n=20, 25%: 1800 train / 600 val / 2400 test ===
  + [4, 9] -> [4, 9]  val acc 0.5550
  + [10, 12] -> [4, 9, 10, 12]  val acc 0.6117
  + [5, 15] -> [4, 5, 9, 10, 12, 15]  val acc 0.6583
  + [1, 7] -> [1, 4, 5, 7, 9, 10, 12, 15]  val acc 0.7350
  + [0, 2] -> [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]  val acc 0.8700
  selected 10 features: [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]
    mlp     small   lr 0.001  val 0.7317  (best epoch 290)
    mlp     small   lr 0.003  val 0.8167  (best epoch 250)
    mlp     small   lr 0.01   val 0.8467  (best epoch 224)
    mlp     medium  lr 0.001  val 0.9100  (best epoch 187)
    mlp     medium  lr 0.003  val 0.9150  (best epoch 106)
    mlp     medium  lr 0.01   val 0.9200  (best epoch 58)
    mlp     large   lr 0.001  val 0.9033  (best epoch 82)
    mlp     large   lr 0.003  val 0.9283  (best epoch 76)
    mlp     large   lr 0.01   val 0.9317  (best epoch 84)
  best mlp     {'arch': 'large', 'hidden': (128, 128, 128), 'lr': 0.01, 'epoch': 84}  val 0.9317  test 

## Summary

In [6]:
print(f"{'n':>3} {'frac':>5} {'model':<7} {'arch':<7} {'lr':>6} {'val':>7} {'test':>7}")
for r in results:
    p = r["params"]
    print(f"{r['n']:>3} {r['frac']:>5.0%} {r['model']:<7} {p['arch']:<7} {p['lr']:>6g} "
          f"{r['val']:>7.4f} {r['test']:>7.4f}")

  n  frac model   arch        lr     val    test
 20   25% mlp     large     0.01  0.9317  0.9433
 20   25% resmlp  large    0.003  0.9483  0.9513
 20   50% mlp     large     0.01  0.9583  0.9579
 20   50% resmlp  large     0.01  0.9583  0.9583
 20  100% mlp     medium   0.001  0.9575  0.9604
 20  100% resmlp  medium    0.01  0.9571  0.9604
 30   25% mlp     large    0.003  0.9490  0.9438
 30   25% resmlp  large     0.01  0.9560  0.9532
 30   50% mlp     medium   0.001  0.9555  0.9572
 30   50% resmlp  medium    0.01  0.9575  0.9580
 30  100% mlp     medium   0.001  0.9565  0.9580
 30  100% resmlp  small    0.003  0.9565  0.9580
 40   25% mlp     large     0.01  0.9469  0.9581
 40   25% resmlp  medium    0.01  0.9513  0.9623
 40   50% mlp     medium   0.003  0.9566  0.9606
 40   50% resmlp  small    0.001  0.9563  0.9622
 40  100% mlp     medium   0.001  0.9569  0.9631
 40  100% resmlp  small    0.001  0.9569  0.9631


## Loading a saved model

In [7]:
def load_model(family, n, frac):
    """Rebuild a saved best model and return (net, feature indices)."""
    ckpt = torch.load(os.path.join(WEIGHT_ROOT, family, f"n{n}", f"{round(frac * 100)}pct", "model.pt"))
    cls, archs = FAMILIES[family]
    net = cls(len(ckpt["features"]), **dict(archs)[ckpt["params"]["arch"]])
    net.load_state_dict(ckpt["state_dict"])
    return net.eval(), ckpt["features"]


# sanity check: the reloaded models reproduce the reported test accuracy
for r in results:
    net, cols = load_model(r["model"], r["n"], r["frac"])
    _, _, _, _, X_test, y_test = split(*load(f"kryptonite-{r['n']}"))
    assert abs(accuracy(net, *to_tensors(binarise(X_test)[:, cols], y_test)) - r["test"]) < 1e-9
print("all saved models reload and reproduce their test accuracy")

all saved models reload and reproduce their test accuracy
